In [ ]:
from queue import PriorityQueue

class Node:
    def __init__(self, position, parent=None):
        self.position = position   
        self.parent = parent       
        self.h = 0          # Heuristic value
        self.f = 0          # Total cost

    def __lt__(self, other):     
        return self.f < other.f


def heuristic(current_pos, end_pos):
    # Manhattan distance heuristic
    return abs(current_pos[0] - end_pos[0]) + abs(current_pos[1] - end_pos[1])

# Best First Search to One Goal
def best_first_search_single_goal(maze, start, goal):
    rows, cols = len(maze), len(maze[0])  

    start_node = Node(start)              
    frontier = PriorityQueue()            
    frontier.put(start_node)              
    visited = set()                       

    while not frontier.empty():
        current_node = frontier.get()       # Remove node with smallest f value
        current_pos = current_node.position

        if current_pos == goal:             # If we reached the goal, reconstruct path
            path = []
            while current_node:
                path.append(current_node.position)
                current_node = current_node.parent
            return path[::-1]               # Reverse the path to start from the start position

        visited.add(current_pos)            # Mark current position as visited

        # Generate adjacent nodes
        for dx, dy in [(1,0), (-1,0), (0,1), (0,-1)]:    
            new_pos = (current_pos[0] + dx, current_pos[1] + dy)

            if (0 <= new_pos[0] < rows and 0 <= new_pos[1] < cols and maze[new_pos[0]][new_pos[1]] != 1 and new_pos not in visited):
                new_node = Node(new_pos, current_node)        
                new_node.h = heuristic(new_pos, goal)        
                new_node.f = new_node.h          # f(n) = h(n) (Greedy Best First)
                frontier.put(new_node)                        
    return None         # No path found                                       

# Enhanced Multi-Goal Search
def best_first_search_multiple_goals(maze, start, goals):
    current_start = start
    total_path = []
    remaining_goals = goals.copy()      # Copy all the goals that have to be reached

    while remaining_goals:
        nearest_goal = min( remaining_goals, key=lambda g: heuristic(current_start, g))     # Choose nearest goal using heuristic

        path = best_first_search_single_goal(maze, current_start, nearest_goal)

        if path is None:
            print("One of the goals is unreachable!")
            return None

        # Avoid duplicating start node in path
        if total_path:
            total_path.extend(path[1:])
        else:
            total_path.extend(path)

        current_start = nearest_goal
        remaining_goals.remove(nearest_goal)

    return total_path

# Example Maze with Multiple Goals
maze = [
[0, 0, 1, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 1, 0, 1],
[0, 0, 1, 0, 0],
[0, 0, 0, 1, 0]
]
start = (0, 0)
goals = [(3, 4), (2, 3)]   # Multiple goals

path = best_first_search_multiple_goals(maze, start, goals)

if path:
    print("Path found:")
    print(path)
    print("Total Steps:", len(path)-1)
else:
    print("No path found.")


Path found:
[(0, 0), (1, 0), (1, 1), (1, 2), (1, 3), (2, 3), (3, 3), (3, 4)]
Total Steps: 7


In [5]:
import random

class Node:
    def __init__(self, position):
        self.position = position
        self.parent = None
        self.g = float('inf')
        self.h = 0
        self.f = float('inf')

    def __lt__(self, other):
        return self.f < other.f

# Heuristic (Manhattan)
def heuristic(a, b):
    return abs(a[0]-b[0]) + abs(a[1]-b[1])

def dynamic_cost():
    return random.randint(1, 5)

def dynamic_a_star(maze, start, goal):

    rows, cols = len(maze), len(maze[0])

    nodes = {}
    open_list = []
    closed = set()

    start_node = Node(start)
    start_node.g = 0
    start_node.h = heuristic(start, goal)
    start_node.f = start_node.h

    nodes[start] = start_node
    open_list.append(start_node)

    while open_list:

        open_list.sort()
        current = open_list.pop(0)

        if current.position == goal:
            path = []
            while current:
                path.append(current.position)
                current = current.parent
            return path[::-1]

        closed.add(current.position)

        for dx, dy in [(1,0), (-1,0), (0,1), (0,-1)]:
            new_pos = (current.position[0]+dx, current.position[1]+dy)

            if (0 <= new_pos[0] < rows and
                0 <= new_pos[1] < cols and
                maze[new_pos[0]][new_pos[1]] != 1):

                cost = dynamic_cost()  # cost changes dynamically

                if new_pos not in nodes:
                    nodes[new_pos] = Node(new_pos)

                neighbor = nodes[new_pos]

                if new_pos in closed:
                    continue

                new_g = current.g + cost

                if new_g < neighbor.g:
                    neighbor.g = new_g
                    neighbor.h = heuristic(new_pos, goal)
                    neighbor.f = neighbor.g + neighbor.h
                    neighbor.parent = current

                    if neighbor not in open_list:
                        open_list.append(neighbor)

    return None

maze = [
    [0,0,0,1,1],
    [0,1,0,1,0],
    [0,0,1,0,1],
    [0,0,0,0,1],
    [0,0,0,1,0],
]

start = (0,0)
goal = (3,3)

path = dynamic_a_star(maze, start, goal)

if path:
    print("Optimal Path Found:")
    print(path)
else:
    print("No Path Found.")


Optimal Path Found:
[(0, 0), (1, 0), (2, 0), (2, 1), (3, 1), (3, 2), (3, 3)]


In [11]:
from queue import PriorityQueue

# graph where each node can have <= 2 children
graph = {
    'S': [('A', 2), ('B', 4)],
    'A': [('C', 3), ('D', 5)],
    'B': [('E', 6)],
    'C': [('F', 2)],
    'D': [],
    'E': [('G', 3)],
    'F': [],
    'G': []
}

heuristics = {
    'S': 6, 'A': 4, 'B': 7, 'C': 2,
    'D': 8, 'E': 5, 'F': 1, 'G': 9
}

deliveries = {
    'C': 6,   # Should succeed
    'F': 7,   # Should fail (tight path)
    'G': 9,   # Should fail (long path)
    'D': 5    # Should fail
}


def greedy_best_first_search(start, graph, heuristics, deliveries):
    # Sort delivery nodes by deadline (earliest first)
    sorted_deliveries = sorted(deliveries.items(), key=lambda x: x[1])
    
    current_node = start
    total_time = 0
    
    print("Starting deliveries from node:", start)
    
    for goal, deadline in sorted_deliveries:
        print(f"\nNext delivery node: {goal} (Deadline: {deadline})")
        
        # Priority Queue for GBFS
        frontier = PriorityQueue()
        frontier.put((heuristics[current_node], [current_node]))
        visited = set()
        path_found = False
        
        while not frontier.empty():
            _, path = frontier.get()
            node = path[-1]
            
            if node == goal:
                path_found = True
                # Add path cost (sum of edge weights)
                for i in range(len(path)-1):
                    for neighbor, cost in graph[path[i]]:
                        if neighbor == path[i+1]:
                            total_time += cost
                print(f"Path taken: {' -> '.join(path)}, Time spent: {total_time}")
                
                # Check delivery success
                if total_time <= deadline:
                    print(f"\nDelivery to {goal} SUCCESSFUL")
                else:
                    print(f"\nDelivery to {goal} FAILED (Exceeded deadline)")
                
                # Move current position to this node
                current_node = goal
                break
            
            if node not in visited:
                visited.add(node)
                for neighbor, _ in graph[node]:
                    if neighbor not in visited:
                        # Greedy: priority = heuristic value
                        frontier.put((heuristics[neighbor], path + [neighbor]))
        
        if not path_found:
            print(f"\nDelivery to {goal} FAILED (No path found)")

# Run the simulation
greedy_best_first_search('S', graph, heuristics, deliveries)

Starting deliveries from node: S

Next delivery node: D (Deadline: 5)
Path taken: S -> A -> D, Time spent: 7

Delivery to D FAILED (Exceeded deadline)

Next delivery node: C (Deadline: 6)

Delivery to C FAILED (No path found)

Next delivery node: F (Deadline: 7)

Delivery to F FAILED (No path found)

Next delivery node: G (Deadline: 9)

Delivery to G FAILED (No path found)
